# Hands-on 27 (Bab 11): Ekspor Model Sederhana ke Format Ringan

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 11.

Dua jalur deployment edge yang realistis:
- **Jalur A:** Random Forest scikit-learn -> joblib (untuk Raspberry Pi dengan Python).
- **Jalur B:** MLP kecil Keras -> .keras -> TensorFlow Lite (untuk inferensi edge).

## Tujuan

Pada latihan ini, pembaca akan:

- melatih model sederhana di laptop Ubuntu,
- mengekspor model ke format yang lebih ringan atau lebih mudah dideploy,
- dan menyiapkan model agar siap dipakai pada inferensi lokal.

Inti latihan ini adalah memahami bahwa training dan deployment adalah dua tahap yang berbeda. Model yang nyaman dilatih di laptop belum tentu langsung ideal untuk edge. Karena itu, perlu ada proses pengemasan ulang agar model lebih praktis dipakai di perangkat target. Konversi model ringan dan TensorFlow Lite secara praktis adalah fokus utama bab ini.

## 2. Contoh implementasi nyata

Contoh yang realistis untuk tahap ini:

- model klasifikasi normal / warning / fault dari data ruangan,
- model prediksi sederhana suhu atau energi,
- model anomaly score ringan untuk edge gateway.

Use case seperti ini cocok karena:

- ukuran model relatif kecil,
- input biasanya berbentuk tabular,
- dan inferensi bisa dilakukan cepat di edge.

## 3. Strategi yang dipakai

Pada hands-on ini kita akan menunjukkan dua jalur yang realistis:

Jalur A — model scikit-learn

- latih model ringan
- simpan dengan joblib
- deploy di Raspberry Pi dengan Python

Jalur B — model Keras ringan

- latih model kecil
- konversi ke TensorFlow Lite
- jalankan inferensi memakai interpreter TFLite

Pendekatan ini sangat realistis untuk buku praktis, karena tidak semua kasus perlu memaksa model deep learning. Referensi juga menempatkan scikit-learn,

TensorFlow, dan Keras sebagai ekosistem open source yang relevan untuk pembelajaran ML.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /downloads/iot-ml-book


## 1. Jalur A: Ekspor scikit-learn (joblib)

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/export_sklearn_model.py"], capture_output=True, text=True)
print(r.stdout)

Model scikit-learn berhasil disimpan ke: models/room_status_rf.joblib



## 2. Jalur B: Keras -> TFLite

In [3]:
r = subprocess.run([sys.executable, "scripts/export_keras_to_tflite.py"], capture_output=True, text=True)
print(r.stdout)

Epoch 1/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 22s 481ms/step - accuracy: 0.3750 - loss: 9.9634
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 730us/step - accuracy: 0.9199 - loss: 1.0015 
Epoch 2/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 4s 85ms/step - accuracy: 0.9688 - loss: 0.9523
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 706us/step - accuracy: 0.9775 - loss: 0.2311
Epoch 3/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.9688 - loss: 0.0332
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 682us/step - accuracy: 0.9801 - loss: 0.2068
Epoch 4/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.9688 - loss: 0.1355
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 672us/step - accuracy: 0.9795 - loss: 0.2068
Epoch 5/10

 1/48 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accur

## 3. Verifikasi Inferensi dari File Model

In [4]:
import joblib
import numpy as np
import pandas as pd

# Inferensi ulang dari model joblib
model = joblib.load("models/room_status_rf.joblib")
X_test = pd.read_csv("data/processed/classification_status/X_test.csv")
print("[INFO] Prediksi joblib (5 baris):", list(model.predict(X_test.head())))

# Inferensi ulang dari model TFLite (batch dim default 1: resize dulu ke 5 baris)
import tensorflow as tf
interpreter = tf.lite.Interpreter(model_path="models/room_binary.tflite")
Xb = pd.read_csv("data/processed/classification_room/X_test.csv").values.astype("float32")[:5]
inp_detail = interpreter.get_input_details()[0]
interpreter.resize_tensor_input(inp_detail["index"], Xb.shape)
interpreter.allocate_tensors()
interpreter.set_tensor(inp_detail["index"], Xb)
interpreter.invoke()
out = interpreter.get_output_details()[0]["index"]
print("[INFO] Probabilitas TFLite (5 baris):", interpreter.get_tensor(out).flatten().round(3).tolist())

[INFO] Prediksi joblib (5 baris): ['normal', 'normal', 'normal', 'normal', 'normal']


I0000 00:00:1793596494.809980    2532 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793596494.810369    2532 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793596494.838045    2532 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1793596495.751815    2532 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793596495.752119    2532 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


[INFO] Probabilitas TFLite (5 baris): [0.9990000128746033, 0.0, 0.0, 0.0, 0.0]


/home/sandbox/.local/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## Panduan Lengkap dari Buku

_Bagian ini dikutip dari buku PDF (sumber kebenaran praktikum) dan diformat ulang agar mudah dibaca. Kode asli buku tidak diubah; bagian instalasi/persiapan lingkungan diringkas karena sudah dijelaskan di README._

### 5. Ekspor model scikit-learn ringan

Kita pakai contoh model klasifikasi ruangan.

Buat file scripts/export_sklearn_model.py:

```python
from pathlib import Path
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
DATA_DIR = Path("data/processed/classification_status")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
model_path = MODEL_DIR / "room_status_rf.joblib"
joblib.dump(model, model_path)
print("Model scikit-learn berhasil disimpan ke:", model_path)
```

Jalankan:

```python
python scripts/export_sklearn_model.py
```

Mengapa ini sudah “ringan”?

Untuk Raspberry Pi atau mini PC Linux, model scikit-learn kecil seperti Random Forest yang dibatasi kedalamannya sering sudah cukup praktis. Tidak semua deployment edge harus langsung memakai TensorFlow Lite. Ini sesuai dengan semangat bab: realistis, bukan memaksa semua hal menjadi TinyML.

### 6. Ekspor model Keras ke TensorFlow Lite

Sekarang contoh jalur neural network kecil.

Buat file scripts/export_keras_to_tflite.py:

```python
from pathlib import Path
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
DATA_DIR = Path("data/processed/classification_room")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
X_train = pd.read_csv(DATA_DIR /
"X_train.csv").values.astype("float32")
y_train = pd.read_csv(DATA_DIR /
"y_train.csv").squeeze().values.astype("float32")
model = Sequential([
    Dense(16, activation="relu", input_shape=(X_train.shape[1],)),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])
model.compile(optimizer="adam", loss="binary_crossentropy",
metrics=["accuracy"])
model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)
keras_path = MODEL_DIR / "room_binary.keras"
model.save(keras_path)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
tflite_path = MODEL_DIR / "room_binary.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)
print("Model Keras disimpan ke:", keras_path)
print("Model TFLite disimpan ke:", tflite_path)
print("Ukuran file TFLite:", tflite_path.stat().st_size, "bytes")
```

Jalankan:

```python
python scripts/export_keras_to_tflite.py
```

### 7. Apa yang harus dipahami pembaca?

Pada tahap ini pembaca belajar bahwa:

- model training bisa tetap dilakukan di laptop,
- model lalu disimpan ke format yang cocok untuk deployment,
- joblib cukup realistis untuk model klasik di Raspberry Pi,
- TensorFlow Lite adalah jalur praktis untuk model neural network ringan.

Ini konsisten dengan arsitektur edge modern, di mana edge dipakai untuk inferensi dan cloud/laptop dipakai untuk training dan eksperimen.

## Penutup

Training dan deployment adalah dua tahap berbeda. Model joblib (sekitar 425 KB) siap
untuk Raspberry Pi dengan Python biasa; model TFLite hanya sekitar 3.9 KB dan bisa
berjalan dengan interpreter ringan tanpa TensorFlow penuh. Keduanya terverifikasi bisa
dimuat ulang dan dipakai inferensi dari file hasil ekspor di `models/`.